In [4]:
import os
from pathlib import Path
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-large"

CONTRACT_DIR = Path("contracts")  # Put authorized PDF/DOCX/TXT files here
CHROMA_DIR = Path("LegalContractChroma")

Enter your OpenAI API key:  ········


## Load Documents

In [7]:
from langchain_core.documents import Document
import fitz
from docx import Document as DocxDocument

def load_contract_file(path: Path) -> list[Document]:
    ext = path.suffix.lower()
    docs = []

    if ext == ".pdf":
        with fitz.open(path) as pdf:
            for page_no, page in enumerate(pdf, start=1):
                text = page.get_text("text").strip()
                if text:
                    docs.append(Document(
                        page_content=text,
                        metadata={"source": path.name, "page": page_no, "file_type": "pdf"}
                    ))

    elif ext == ".docx":
        docx = DocxDocument(path)
        text = "\n".join(p.text for p in docx.paragraphs if p.text.strip())
        if text.strip():
            docs.append(Document(
                page_content=text,
                metadata={"source": path.name, "page": None, "file_type": "docx"}
            ))

    elif ext == ".txt":
        text = path.read_text(encoding="utf-8", errors="replace").strip()
        if text:
            docs.append(Document(
                page_content=text,
                metadata={"source": path.name, "page": None, "file_type": "txt"}
            ))

    else:
        print(f"Skipping unsupported file: {path.name}")

    return docs


def load_contract_directory(directory: Path) -> list[Document]:
    if not directory.exists():
        directory.mkdir(parents=True, exist_ok=True)
        print(f"Created {directory.resolve()}. Add authorized contract files, then rerun this cell.")
        return []

    all_docs = []
    for path in directory.iterdir():
        if path.is_file():
            try:
                all_docs.extend(load_contract_file(path))
            except Exception as exc:
                print(f"Could not read {path.name}: {exc}")
    return all_docs


documents = load_contract_directory(CONTRACT_DIR)
print(f"Loaded {len(documents)} document sections/pages.")
if documents:
    print(documents[0].metadata)
    print(documents[0].page_content[:500])

Created C:\Users\Akankssha Adepu\contracts. Add authorized contract files, then rerun this cell.
Loaded 0 document sections/pages.


C:\Users\Akankssha Adepu\anaconda3\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## Chunking

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "; ", " ", ""]
)

chunks = splitter.split_documents(documents)
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print(f"Created {len(chunks)} chunks.")

Created 0 chunks.


In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = None
if chunks:
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="legal_contracts",
        persist_directory=str(CHROMA_DIR)
    )
    print("Vector store created.")
else:
    print("No chunks yet. Add contracts to the contracts/ directory and rerun cells 2–4.")

No chunks yet. Add contracts to the contracts/ directory and rerun cells 2–4.


In [14]:
pip install langchain_chroma

  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached pybase64-1.5.1-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.45.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.45.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached grpcio-1.85.0-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached kubernetes-37.0.1-py2.py3-none-any.whl.metadata (1.9 kB)
  Using cached mmh3-5.3.1-cp314-cp314-win_amd64.whl.metadata (14 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached requests_oauthlib-2.0.0-py2.p

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [19]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

retriever = None
if chunks and vector_store is not None:
    dense_retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 12, "lambda_mult": 0.65}
    )
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 5

    retriever = EnsembleRetriever(
        retrievers=[dense_retriever, bm25_retriever],
        weights=[0.6, 0.4]
    )
    print("Hybrid retriever ready.")

C:\Users\Akankssha Adepu\AppData\Local\Temp\ipykernel_1480\1174191240.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [20]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

SYSTEM_INSTRUCTIONS = """
You are a careful legal-contract document review assistant.
Use ONLY the supplied contract excerpts to make claims about this contract.
Do not invent clauses, parties, dates, amounts, governing law, or obligations.
If the excerpts do not support an answer, say: "Insufficient evidence in the retrieved contract excerpts."
Separate:
1. What the contract explicitly says
2. Potential issue to review
3. Questions or missing information
Cite source filename and page number when metadata is available.
Do not present a risk label as a definitive legal conclusion.
Do not claim to be a lawyer or replace qualified legal advice.
"""

def format_context(docs):
    parts = []
    for d in docs:
        source = d.metadata.get("source", "unknown source")
        page = d.metadata.get("page")
        page_text = f", page {page}" if page is not None else ""
        parts.append(f"[Source: {source}{page_text}]\n{d.page_content}")
    return "\n\n---\n\n".join(parts)


def baseline_rag(question: str, k: int = 5) -> dict:
    if retriever is None:
        raise RuntimeError("No retriever is ready. Add contract files and run ingestion cells first.")

    retrieved = retriever.invoke(question)
    context = format_context(retrieved)
    prompt = f"""{SYSTEM_INSTRUCTIONS}

Retrieved contract excerpts:
{context}

User question: {question}

Answer with concise headings:
- Answer based on the contract
- Evidence and citations
- Potential issues to review (if relevant)
- Limitations / insufficient evidence (if relevant)
"""
    response = llm.invoke(prompt)
    return {
        "question": question,
        "answer": response.content,
        "contexts": [d.page_content for d in retrieved],
        "documents": retrieved
    }

In [21]:
# Example: edit this question to match a clause in your uploaded contract.
question = "What are the termination rights and required notice period?"
if retriever is not None:
    result_baseline = baseline_rag(question)
    print(result_baseline["answer"])
else:
    print("Add a contract to the contracts/ folder, then rerun the ingestion and retrieval cells.")

Add a contract to the contracts/ folder, then rerun the ingestion and retrieval cells.


In [22]:
import pandas as pd

EVAL_PATH = Path("evaluation_questions.csv")

if EVAL_PATH.exists():
    eval_df = pd.read_csv(EVAL_PATH)
    print(eval_df.head())
else:
    eval_df = pd.DataFrame([
        {
            "question": "What notice is required to terminate the agreement?",
            "reference_answer": "REPLACE with a manually verified answer from your contract.",
            "expected_source": "REPLACE_WITH_FILENAME.pdf",
            "expected_page": 1
        }
    ])
    eval_df.to_csv(EVAL_PATH, index=False)
    print(f"Created template at {EVAL_PATH.resolve()}. Replace the placeholder with verified examples.")

Created template at C:\Users\Akankssha Adepu\evaluation_questions.csv. Replace the placeholder with verified examples.


In [23]:
from time import perf_counter

def evaluate_retrieval(question, expected_source=None, expected_page=None, k=5):
    if retriever is None:
        raise RuntimeError("Retriever not ready.")
    start = perf_counter()
    retrieved = retriever.invoke(question)
    elapsed = perf_counter() - start

    source_hit = any(
        d.metadata.get("source") == expected_source
        for d in retrieved
    ) if expected_source and expected_source != "REPLACE_WITH_FILENAME.pdf" else None

    page_hit = any(
        d.metadata.get("source") == expected_source
        and str(d.metadata.get("page")) == str(expected_page)
        for d in retrieved
    ) if expected_source and expected_page is not None and expected_source != "REPLACE_WITH_FILENAME.pdf" else None

    return {
        "question": question,
        "retrieved_count": len(retrieved),
        "expected_source_hit": source_hit,
        "expected_source_page_hit": page_hit,
        "retrieval_seconds": round(elapsed, 3),
        "retrieved_sources": [
            f"{d.metadata.get('source')} page={d.metadata.get('page')}"
            for d in retrieved
        ]
    }


retrieval_rows = []
if retriever is not None and not eval_df.empty:
    for _, row in eval_df.iterrows():
        retrieval_rows.append(evaluate_retrieval(
            row["question"],
            row.get("expected_source"),
            row.get("expected_page")
        ))
    retrieval_results = pd.DataFrame(retrieval_rows)
    display(retrieval_results)
else:
    print("Evaluation will run after contracts and verified evaluation examples are added.")

Evaluation will run after contracts and verified evaluation examples are added.


In [24]:
# Example report template. Populate only from measured results.
report_template = pd.DataFrame([
    {"system": "Baseline Hybrid RAG", "retrieval_recall_at_k": None, "expert_correctness": None, "faithfulness": None, "citation_precision": None, "latency_seconds": None, "cost_per_question": None},
    {"system": "Fine-tuned model + same Hybrid RAG", "retrieval_recall_at_k": None, "expert_correctness": None, "faithfulness": None, "citation_precision": None, "latency_seconds": None, "cost_per_question": None},
])
display(report_template)

,system,retrieval_recall_at_k,expert_correctness,faithfulness,citation_precision,latency_seconds,cost_per_question
0,Baseline Hybrid RAG,None,None,None,None,None,None
1,Fine-tuned model + same Hybrid RAG,None,None,None,None,None,None
